# Avance 2: Ingeniería de Características y Modelamiento Supervisado

**Objetivo:** Construir y evaluar modelos predictivos para determinar si un cliente pagará o no a tiempo (`Pago_atiempo`).

**Contenido:**
1. Ingeniería de características (feature engineering)
2. Entrenamiento de modelos supervisados
3. Evaluación y comparación de modelos
4. Selección del mejor modelo

---
## 1. Importar librerías y módulos

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier

from ft_engineering import (
    load_and_clean_data,
    create_features,
    build_preprocessor,
    prepare_data,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    ORDINAL_FEATURES,
)
from model_training_evaluation import build_model, compare_models

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["font.size"] = 11
sns.set_style("whitegrid")

print("Módulos importados correctamente.")

---
## 2. Ingeniería de Características

### 2.1 Carga y limpieza de datos

In [ ]:
df = load_and_clean_data("../Base_de_datos.csv")
print(f"Dataset cargado: {df.shape}")
print(f"\nColumnas: {df.columns.tolist()}")
print(f"\nNulos por columna:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### 2.2 Creación de features derivados

In [ ]:
df = create_features(df)

print("Features derivados creados:")
print(f"  - ratio_cuota_salario:  {df['ratio_cuota_salario'].describe().to_dict()}")
print(f"  - ratio_deuda_salario:  {df['ratio_deuda_salario'].describe().to_dict()}")
print(f"  - ratio_capital_plazo:  {df['ratio_capital_plazo'].describe().to_dict()}")

### 2.3 Clasificación de variables para el pipeline

El `ColumnTransformer` aplica transformaciones diferenciadas según el tipo de variable:

| Pipeline | Variables | Transformaciones |
|----------|-----------|------------------|
| **numeric** | Variables continuas y discretas | `SimpleImputer(median)` |
| **categoric** | tipo_credito, tipo_laboral | `SimpleImputer(most_frequent)` → `OneHotEncoder` |
| **categoric_ordinal** | tendencia_ingresos | `SimpleImputer(most_frequent)` → `OrdinalEncoder` |

In [ ]:
print("Variables numéricas:", NUMERIC_FEATURES)
print(f"\nVariables categóricas nominales: {CATEGORICAL_FEATURES}")
print(f"\nVariables categóricas ordinales: {ORDINAL_FEATURES}")

### 2.4 Preparación de datos (train/test split + pipeline)

In [ ]:
X_train, X_test, y_train, y_test, preprocessor = prepare_data("../Base_de_datos.csv")

print(f"\nForma X_train: {X_train.shape}")
print(f"Forma X_test:  {X_test.shape}")

In [ ]:
# Visualizar la estructura del preprocessor
print("Estructura del ColumnTransformer:")
for name, pipeline, columns in preprocessor.transformers_:
    print(f"\n  [{name}]")
    print(f"    Columnas: {columns}")
    if hasattr(pipeline, 'steps'):
        for step_name, step in pipeline.steps:
            print(f"    → {step_name}: {step.__class__.__name__}")

---
## 3. Entrenamiento de Modelos Supervisados

Se entrenarán los siguientes modelos de clasificación:
1. Regresión Logística
2. Árbol de Decisión
3. Random Forest
4. Gradient Boosting
5. XGBoost

In [ ]:
# Almacenar resultados de todos los modelos
all_results = []

### 3.1 Regresión Logística

In [ ]:
lr_model, lr_metrics = build_model(
    model=LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    model_name="Regresión Logística",
)
all_results.append(lr_metrics)

### 3.2 Árbol de Decisión

In [ ]:
dt_model, dt_metrics = build_model(
    model=DecisionTreeClassifier(class_weight="balanced", random_state=42),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    model_name="Árbol de Decisión",
)
all_results.append(dt_metrics)

### 3.3 Random Forest

In [ ]:
rf_model, rf_metrics = build_model(
    model=RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42, n_jobs=-1,
    ),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    model_name="Random Forest",
)
all_results.append(rf_metrics)

### 3.4 Gradient Boosting

In [ ]:
gb_model, gb_metrics = build_model(
    model=GradientBoostingClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4, random_state=42,
    ),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    model_name="Gradient Boosting",
)
all_results.append(gb_metrics)

### 3.5 XGBoost

In [ ]:
# Calcular scale_pos_weight para manejar desbalance
scale_pos = (y_train == 0).sum() / (y_train == 1).sum()

xgb_model, xgb_metrics = build_model(
    model=XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        scale_pos_weight=1/scale_pos, eval_metric="logloss",
        random_state=42, n_jobs=-1,
    ),
    X_train=X_train, y_train=y_train,
    X_test=X_test, y_test=y_test,
    model_name="XGBoost",
)
all_results.append(xgb_metrics)

---
## 4. Comparación y Selección de Modelos

In [ ]:
df_results = compare_models(all_results)

In [ ]:
df_results

---
## 5. Conclusiones del Avance 2

### Resumen:
1. Se implementó el pipeline de **ingeniería de características** con `ColumnTransformer`, que procesa de forma diferenciada variables numéricas, categóricas nominales y categóricas ordinales.
2. Se crearon **3 features derivados** (ratios financieros) que capturan la carga financiera y nivel de endeudamiento del cliente.
3. Se entrenaron **5 modelos supervisados** de clasificación: Regresión Logística, Árbol de Decisión, Random Forest, Gradient Boosting y XGBoost.
4. Se utilizaron las funciones reutilizables `build_model` y `summarize_classification` para automatizar el flujo de entrenamiento y evaluación.
5. Se generaron **gráficos comparativos** (barras y heatmap) y una **tabla resumen** de métricas para seleccionar el mejor modelo.

### Notas:
- Dado el desbalance de clases (95% vs 5%), se utilizó `class_weight="balanced"` y `scale_pos_weight` en los modelos que lo soportan.
- La métrica principal de selección es el **F1-Score**, complementada con el **AUC-ROC**, ya que en problemas desbalanceados el accuracy puede ser engañoso.